# Packet P-037 — Cross-Validation Strategy Comparison

**Decision question:** Does the random CV of 0.289 overstate performance because it leaks family-level signal between train and test? Is leave-one-family-out a more honest measure?

**Fastest falsifier:** If random, stratified, and LOGO CV tau-b are all within 0.02, CV strategy doesn't matter.

**Success:** Stratified CV within 0.02 of random (0.289).
**Kill:** Leave-one-family-out tau-b < 0.10 — model doesn't generalize across families.

In [1]:
"""Cell 1 — Load data, classify families, run 3 CV strategies."""
import pandas as pd
import numpy as np
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import KFold, StratifiedKFold, LeaveOneGroupOut
from scipy.stats import kendalltau
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('perovskite_stability_clean.csv')
FEATURES = [
    'Perovskite_band_gap', 'Pb', 'Sn', 'I', 'Br', 'Cl',
    'MA', 'FA', 'Cs',
    'first_Prvskt_annealing_temperature', 'first_Prvskt_thermal_annealing_time',
    'Perovskite_thickness', 'Cell_area_measured',
    'JV_default_Voc', 'JV_default_Jsc', 'JV_default_FF'
]
TARGET = 'Stability_PCE_T80'
ET_PARAMS = dict(n_estimators=700, max_features='sqrt', min_samples_split=3,
                 min_samples_leaf=1, bootstrap=False, random_state=42)

X = df[FEATURES].fillna(0).values
y = np.log1p(df[TARGET].values)

def classify_family(row):
    ma, fa, cs = row["MA"] > 0, row["FA"] > 0, row["Cs"] > 0
    if ma and not fa and not cs: return "Pure MA"
    elif fa and not ma and not cs: return "Pure FA"
    elif ma and fa and not cs: return "MA-FA mixed"
    elif fa and cs and not ma: return "FA-Cs"
    elif ma and fa and cs: return "Triple cation"
    else: return "Other"

df["family"] = df.apply(classify_family, axis=1)
families = df["family"].values
family_labels, family_codes = np.unique(families, return_inverse=True)

print("Family distribution:")
for i, f in enumerate(family_labels):
    print(f"  {f}: {(family_codes == i).sum()}")

# --- Strategy 1: Random 5-fold CV (baseline) ---
random_taus = []
kf = KFold(n_splits=5, shuffle=True, random_state=42)
for tr, te in kf.split(X):
    m = ExtraTreesRegressor(**ET_PARAMS); m.fit(X[tr], y[tr])
    tau, _ = kendalltau(y[te], m.predict(X[te]))
    random_taus.append(tau)

# --- Strategy 2: Stratified 5-fold CV (by family) ---
strat_taus = []
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for tr, te in skf.split(X, family_codes):
    m = ExtraTreesRegressor(**ET_PARAMS); m.fit(X[tr], y[tr])
    tau, _ = kendalltau(y[te], m.predict(X[te]))
    strat_taus.append(tau)

# --- Strategy 3: Leave-One-Family-Out ---
logo_taus = []
logo_families = []
logo = LeaveOneGroupOut()
for tr, te in logo.split(X, y, family_codes):
    if len(te) < 10:
        continue
    m = ExtraTreesRegressor(**ET_PARAMS); m.fit(X[tr], y[tr])
    tau, _ = kendalltau(y[te], m.predict(X[te]))
    logo_taus.append(tau)
    fam_name = family_labels[family_codes[te[0]]]
    logo_families.append(fam_name)
    print(f"  LOGO fold: held out {fam_name} (n={len(te)}), tau-b = {tau:.4f}")

print(f"\n{'=' * 60}")
print("CV STRATEGY COMPARISON")
print(f"{'=' * 60}")
print(f"{'Strategy':<25} {'Mean tau-b':>11} {'Std':>8}")
print("-" * 45)
print(f"{'Random 5-fold':<25} {np.mean(random_taus):>11.4f} {np.std(random_taus):>8.4f}")
print(f"{'Stratified 5-fold':<25} {np.mean(strat_taus):>11.4f} {np.std(strat_taus):>8.4f}")
print(f"{'Leave-one-family-out':<25} {np.mean(logo_taus):>11.4f} {np.std(logo_taus):>8.4f}")

Family distribution:
  FA-Cs: 44
  MA-FA mixed: 197
  Other: 88
  Pure FA: 50
  Pure MA: 967
  Triple cation: 197


  LOGO fold: held out FA-Cs (n=44), tau-b = -0.0488


  LOGO fold: held out MA-FA mixed (n=197), tau-b = -0.0724


  LOGO fold: held out Other (n=88), tau-b = 0.0566


  LOGO fold: held out Pure FA (n=50), tau-b = 0.1420


  LOGO fold: held out Pure MA (n=967), tau-b = 0.0783


  LOGO fold: held out Triple cation (n=197), tau-b = -0.1264

CV STRATEGY COMPARISON
Strategy                   Mean tau-b      Std
---------------------------------------------
Random 5-fold                  0.2937   0.0145
Stratified 5-fold              0.3000   0.0443
Leave-one-family-out           0.0049   0.0939


In [2]:
"""Cell 2 — Evaluate and save."""

random_mean = np.mean(random_taus)
strat_mean = np.mean(strat_taus)
logo_mean = np.mean(logo_taus)

strat_delta = abs(strat_mean - random_mean)
logo_delta = random_mean - logo_mean

print("── Evaluation ──\n")
print(f"Random vs Stratified delta: {strat_delta:.4f}")
print(f"Random vs LOGO delta: {logo_delta:+.4f}")

# LOGO per-family detail
print(f"\n── LOGO per-family ──")
for fam, tau in zip(logo_families, logo_taus):
    marker = "✓" if tau >= 0.10 else "✗"
    print(f"  {fam:<20} tau-b={tau:.4f} {marker}")

# Status
if strat_delta <= 0.02 and logo_mean >= 0.10:
    status = "Confirmed"
elif logo_mean < 0.10:
    status = "Negative"
elif strat_delta <= 0.02:
    status = "Promising"
else:
    status = "Negative"

# Save
save_rows = [
    {'strategy': 'Random 5-fold', 'mean_tau': random_mean, 'std_tau': np.std(random_taus)},
    {'strategy': 'Stratified 5-fold', 'mean_tau': strat_mean, 'std_tau': np.std(strat_taus)},
    {'strategy': 'Leave-one-family-out', 'mean_tau': logo_mean, 'std_tau': np.std(logo_taus)},
]
for fam, tau in zip(logo_families, logo_taus):
    save_rows.append({'strategy': f'LOGO_{fam}', 'mean_tau': tau, 'std_tau': 0})

pd.DataFrame(save_rows).to_csv('Packet_P037_CV_Strategy.csv', index=False)
print(f"\nSaved: Packet_P037_CV_Strategy.csv")

print(f"\n{'=' * 60}")
print(f"P-037 STATUS: {status}")
print(f"{'=' * 60}")
if status == "Confirmed":
    print("Random and stratified CV agree. LOGO shows cross-family generalization.")
elif status == "Negative" and logo_mean < 0.10:
    print("LOGO tau-b < 0.10 — model does NOT generalize across families.")
    print("Random CV of 0.289 overstates performance by leaking family signal.")
else:
    print("CV strategies diverge — reporting should use stratified or LOGO.")

── Evaluation ──

Random vs Stratified delta: 0.0063
Random vs LOGO delta: +0.2889

── LOGO per-family ──
  FA-Cs                tau-b=-0.0488 ✗
  MA-FA mixed          tau-b=-0.0724 ✗
  Other                tau-b=0.0566 ✗
  Pure FA              tau-b=0.1420 ✓
  Pure MA              tau-b=0.0783 ✗
  Triple cation        tau-b=-0.1264 ✗

Saved: Packet_P037_CV_Strategy.csv

P-037 STATUS: Negative
LOGO tau-b < 0.10 — model does NOT generalize across families.
Random CV of 0.289 overstates performance by leaking family signal.
